# Multi-Head Attention

This notebook builds on [causal-attention](causal-attention.ipynb). There, we built a single `CausalAttention` module: one set of $W_q$, $W_k$, $W_v$ matrices, producing one set of attention weights and one context vector per token. This is called **single-head attention** -- there's exactly one "point of view" deciding what's relevant.

**Multi-head attention** runs *several* of these attention computations side by side, on the exact same input, each with its own independently-learned weights -- and then combines their results. Each independent attention computation is called a **head**.

**Why would running several heads help?** A single set of $W_q$, $W_k$, $W_v$ matrices has to learn one specific way of measuring "relevance" between tokens, and squeeze all of that judgment into one shared representation. But language has many different *kinds* of relationships at once: a word might need to track its grammatical subject, a pronoun might need to track which noun it refers to, and a word might need to track nearby words that change its meaning -- all at the same time, and all are different in nature. By giving the model several independent heads, each head is free to specialize in a different kind of relationship, since each has its own weights, learned independently. Then, by combining every head's output, the model gets a richer representation that draws on all of these different "perspectives" at once -- similar to asking several different experts to each independently read the same sentence and give their opinion, and then combining every opinion into a final answer.

We'll build multi-head attention two different ways:

1. **The wrapper approach** -- the most intuitive way, quite literally stacking several independent `CausalAttention` modules side by side.
2. **The efficient, split-weights approach** -- a single module that produces the exact same *kind* of result, but computes it with far fewer, larger matrix multiplications, which is dramatically faster in practice (especially on a GPU).

As always, every new idea will be verified with real numbers -- including, at the end, a from-scratch proof that the "efficient" version really does compute exactly the same thing as running several independent heads by hand.

## 1. Setup: Picking Up Where We Left Off

We reuse the same sentence, embeddings, and `CausalAttention` class from the previous notebook, and again simulate a batch of 2 identical sentences (so we can confirm multi-head attention works correctly on batched input, not just a single sentence).

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
sentence = "Hello my name is Ahtesham"
tokens = sentence.split()
num_tokens = len(tokens)
d_in = 3   # input embedding size

token_embeddings = torch.rand(num_tokens, d_in)
batch = torch.stack((token_embeddings, token_embeddings), dim=0)
context_length = batch.shape[1]

print("Tokens:", tokens)
print("Batch shape:", batch.shape, "-> (batch_size, num_tokens, d_in)")

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

print("\nCausalAttention class ready to use as a single 'head'.")

Tokens: ['Hello', 'my', 'name', 'is', 'Ahtesham']
Batch shape: torch.Size([2, 5, 3]) -> (batch_size, num_tokens, d_in)

CausalAttention class ready to use as a single 'head'.


## 2. Approach 1 -- Stacking Multiple Single-Head Modules

The most intuitive way to get multi-head attention: create several independent `CausalAttention` instances, run the *same* input through every one of them, and glue their outputs together side by side (concatenate along the feature dimension). Each `CausalAttention` instance is initialized with its own random weights and learns independently -- that's what makes it a distinct "head."

`nn.ModuleList` is just a Python list that also tells PyTorch "these are all sub-modules of mine" -- so their parameters get tracked, moved to the GPU together, saved together, and so on, exactly as if we'd written them out individually.

In [2]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        # Run x through every head independently, then glue the results together
        # side by side along the last (feature) dimension.
        return torch.cat([head(x) for head in self.heads], dim=-1)

Let's run this with 2 heads, each producing a `d_out=2` context vector. Since we're gluing 2 heads together side by side, we expect a combined output dimension of `2 x 2 = 4`.

In [3]:
torch.manual_seed(123)
d_out = 2  # each individual head's output size
num_heads = 2

mha_wrapper = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0, num_heads=num_heads)
context_vecs_wrapper = mha_wrapper(batch)

print("context_vecs_wrapper.shape:", context_vecs_wrapper.shape,
      "-> (batch_size, num_tokens, d_out * num_heads)")
print()
print(context_vecs_wrapper)

context_vecs_wrapper.shape: torch.Size([2, 5, 4]) -> (batch_size, num_tokens, d_out * num_heads)

tensor([[[-0.8340, -0.3584,  0.7834,  0.6197],
         [-0.7841, -0.2061,  0.7593,  0.5032],
         [-0.7248, -0.1467,  0.7082,  0.4678],
         [-0.6725, -0.1607,  0.6466,  0.4565],
         [-0.6983, -0.1409,  0.6726,  0.4510]],

        [[-0.8340, -0.3584,  0.7834,  0.6197],
         [-0.7841, -0.2061,  0.7593,  0.5032],
         [-0.7248, -0.1467,  0.7082,  0.4678],
         [-0.6725, -0.1607,  0.6466,  0.4565],
         [-0.6983, -0.1409,  0.6726,  0.4510]]], grad_fn=<CatBackward0>)


Look at the shape: `(2, 5, 4)`. The first `2` columns of the last dimension came entirely from head 1's `CausalAttention`, and the last `2` columns came entirely from head 2's -- they were computed completely independently and only joined together at the very last step, via `torch.cat`.

If you ever need the *total* combined output to be a specific size (say, 2 instead of 4), you don't need to change the class at all -- just lower each individual head's `d_out` so that `d_out * num_heads` comes out to the size you want (e.g. `d_out=1` with `num_heads=2` gives a combined size of 2).

## 3. Why This Approach Is Wasteful

This wrapper works correctly, but think about what it's actually doing under the hood: for `num_heads` heads, we run `num_heads` completely separate `nn.Linear` projections for the queries (and `num_heads` separate ones for keys, and again for values). Every one of those is its own independent matrix multiplication.

Modern hardware (especially GPUs) is highly optimized to perform *one big* matrix multiplication far more efficiently than the same total amount of work split across *several smaller* matrix multiplications -- there's a fixed overhead cost to launching each operation, and bigger operations use the hardware's parallelism much more fully. So if we could somehow replace "several small matmuls" with "one big matmul that produces the combined result directly," we'd get the exact same numbers out, just computed faster.

That's exactly what the efficient version below does.

## 4. The Core Idea Behind "Split Weights"

Here's the key mathematical fact that makes the efficient version possible: **multiplying by one big weight matrix, and then slicing the result into column-chunks, gives you exactly the same numbers as multiplying by the corresponding column-slices of that same weight matrix separately.**

That's a mouthful, so let's prove it with actual numbers rather than take it on faith. Suppose we have one big weight matrix `W_big` of shape `(d_in, d_out)`, where `d_out = num_heads * head_dim`. We'll compare:

* **One big matmul, then split:** compute `x @ W_big` (one matrix multiplication), then slice out columns `[0:head_dim]` for "head 1" and columns `[head_dim:2*head_dim]` for "head 2."
* **Several small matmuls:** slice `W_big` itself into the same column-chunks *first* (`W1`, `W2`), and multiply `x` by each chunk separately.

In [4]:
torch.manual_seed(7)
demo_d_in, demo_num_heads, demo_head_dim = 3, 2, 2
demo_d_out = demo_num_heads * demo_head_dim  # 4
demo_x = torch.rand(5, demo_d_in)
W_big = torch.rand(demo_d_in, demo_d_out)

# --- One big matmul, then split into column-chunks ---
Q_big = demo_x @ W_big
Q1_from_split = Q_big[:, :demo_head_dim]
Q2_from_split = Q_big[:, demo_head_dim:]

# --- Several small matmuls, using the matching column-chunks of the SAME weight matrix ---
W1 = W_big[:, :demo_head_dim]
W2 = W_big[:, demo_head_dim:]
Q1_separate = demo_x @ W1
Q2_separate = demo_x @ W2

print("Head 1 -- one-big-matmul-then-split matches several-small-matmuls?",
      torch.allclose(Q1_from_split, Q1_separate))
print("Head 2 -- one-big-matmul-then-split matches several-small-matmuls?",
      torch.allclose(Q2_from_split, Q2_separate))

Head 1 -- one-big-matmul-then-split matches several-small-matmuls? True
Head 2 -- one-big-matmul-then-split matches several-small-matmuls? True


They match exactly. This isn't a coincidence -- it's simply how matrix multiplication works column by column: each output column only ever depends on the matching column of the weight matrix, never on any other column. So grouping columns together into one big matrix, or keeping them apart as several small matrices, produces identical numbers either way. This one fact is the entire justification for "split weights": we can use one big `W_query` (and one big `W_key`, one big `W_value`) to compute *all* heads' queries/keys/values in a single matmul, and only split the *result* into per-head chunks afterward -- with zero loss of correctness.

## 5. Splitting the Result With `.view()`

Now, how do we actually split `Q_big`'s last dimension into `num_heads` chunks of size `head_dim` in PyTorch? By reshaping it with `.view()`.

It's important to understand that `.view()` doesn't do any math and doesn't move any numbers around -- it just relabels the *same* underlying numbers with a new shape. Since `d_out` is exactly `num_heads * head_dim`, reshaping the last dimension from `(d_out,)` into `(num_heads, head_dim)` is always a valid, lossless regrouping: the first `head_dim` numbers become "head 0," the next `head_dim` numbers become "head 1," and so on. That's precisely the same column-chunking we did by hand with slicing above -- `.view()` just does it for us, and does it in a way PyTorch can process very efficiently.

Let's verify that `.view()`-based splitting really does produce the exact same chunks as the manual slicing above.

In [5]:
Q_viewed = Q_big.view(5, demo_num_heads, demo_head_dim)  # (num_tokens, num_heads, head_dim)

print("Q_viewed[:, 0, :] matches the manual head-1 slice?",
      torch.allclose(Q_viewed[:, 0, :], Q1_from_split))
print("Q_viewed[:, 1, :] matches the manual head-2 slice?",
      torch.allclose(Q_viewed[:, 1, :], Q2_from_split))

Q_viewed[:, 0, :] matches the manual head-1 slice? True
Q_viewed[:, 1, :] matches the manual head-2 slice? True


## 6. Rearranging Dimensions for Batched Matrix Multiplication

After `.view()`, our queries (and keys, and values) have shape `(batch, num_tokens, num_heads, head_dim)`. But to compute attention, we need to do a matrix multiplication *within* each head, independently, for every batch item -- in other words, we want PyTorch to treat `(batch, num_heads)` together as "the thing we loop over," and do a normal 2D matrix multiplication using only the `(num_tokens, head_dim)` part.

PyTorch's matrix multiplication (`@`) supports exactly this: when given tensors with more than 2 dimensions, it treats every dimension *except the last two* as a batch of independent 2D matrices to multiply, all at once. So we `.transpose(1, 2)` to swap `num_tokens` and `num_heads`, producing shape `(batch, num_heads, num_tokens, head_dim)` -- now the last two dimensions, `(num_tokens, head_dim)`, are exactly the per-head matrix PyTorch needs, and `(batch, num_heads)` becomes the batch of independent matrices to multiply.

Let's see this "batched matmul" behavior directly with a small example, and confirm it really does give the same result as looping over each head by hand.

In [6]:
torch.manual_seed(123)
a = torch.rand(1, 2, 3, 4)  # shape: (batch=1, num_heads=2, num_tokens=3, head_dim=4)

batched_result = a @ a.transpose(2, 3)  # transpose only the last two dims
print("Batched result shape:", batched_result.shape)
print(batched_result)

Batched result shape: torch.Size([1, 2, 3, 3])
tensor([[[[0.8920, 0.5745, 0.9822],
          [0.5745, 0.7855, 0.7566],
          [0.9822, 0.7566, 1.1331]],

         [[0.3059, 0.5579, 0.7774],
          [0.5579, 1.2809, 1.4843],
          [0.7774, 1.4843, 2.4514]]]])


In [7]:
# Now compute the exact same thing by manually looping over each head.
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T

print("Manual, per-head result for head 1:\n", first_res)
print("\nManual, per-head result for head 2:\n", second_res)

print("\nDoes the batched call match head 1 computed by hand?",
      torch.allclose(batched_result[0, 0], first_res))
print("Does the batched call match head 2 computed by hand?",
      torch.allclose(batched_result[0, 1], second_res))

Manual, per-head result for head 1:
 tensor([[0.8920, 0.5745, 0.9822],
        [0.5745, 0.7855, 0.7566],
        [0.9822, 0.7566, 1.1331]])

Manual, per-head result for head 2:
 tensor([[0.3059, 0.5579, 0.7774],
        [0.5579, 1.2809, 1.4843],
        [0.7774, 1.4843, 2.4514]])

Does the batched call match head 1 computed by hand? True
Does the batched call match head 2 computed by hand? True


They match perfectly. This confirms that `queries @ keys.transpose(2, 3)` (once both have shape `(batch, num_heads, num_tokens, head_dim)`) is really doing the *same* per-head attention-score computation from the single-head `CausalAttention` class -- just for every head, and every batch item, all at once in a single call, instead of writing a Python loop ourselves.

Once we have attention scores of shape `(batch, num_heads, num_tokens, num_tokens)`, everything else is exactly what we already built for single-head causal attention: mask out future positions with `-inf` (the mask is the same `(num_tokens, num_tokens)` grid for every head and every batch item, so it automatically "broadcasts" across those extra dimensions), scale by `sqrt(head_dim)`, run `softmax`, optionally apply dropout, and multiply by the values to get each head's context vectors.

## 7. Putting the Heads Back Together

Once every head has produced its own context vectors, we have a tensor of shape `(batch, num_heads, num_tokens, head_dim)`. To get back to a normal `(batch, num_tokens, d_out)` shape (with `d_out = num_heads * head_dim`), we need to reverse what we did in step 6: `.transpose(1, 2)` back to `(batch, num_tokens, num_heads, head_dim)`, and then flatten the last two dimensions into one with `.view()`.

There's a subtlety here worth understanding: `.transpose()` doesn't physically move any numbers in memory -- it just changes how the tensor's shape is *interpreted*, leaving the underlying data in its original order. `.view()`, on the other hand, needs the underlying data to already be laid out in memory in a specific, "standard" left-to-right order matching the *new* shape it's asked to produce, and after a `.transpose()`, that's no longer true. `.contiguous()` fixes this by making an actual, fresh copy of the data in the correctly-ordered layout, so that `.view()` can safely be called right after. Let's see this play out concretely -- calling `.view()` directly after `.transpose()` should fail, but works fine once we call `.contiguous()` first.

In [8]:
demo_tensor = torch.rand(2, 5, 2, 3)          # (batch, num_tokens, num_heads, head_dim)
demo_transposed = demo_tensor.transpose(1, 2)  # (batch, num_heads, num_tokens, head_dim)

print("Is the transposed tensor contiguous in memory?", demo_transposed.is_contiguous())

try:
    demo_transposed.view(2, 5, 6)
    print("`.view()` worked directly (unexpected).")
except RuntimeError as e:
    print("`.view()` failed directly, as expected:")
    print(" ", e)

demo_contiguous = demo_transposed.contiguous()
print("\nIs it contiguous after calling `.contiguous()`?", demo_contiguous.is_contiguous())

reshaped = demo_contiguous.view(2, 5, 6)
print("`.view()` now works, producing shape:", reshaped.shape)

Is the transposed tensor contiguous in memory? False
`.view()` failed directly, as expected:
  view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

Is it contiguous after calling `.contiguous()`? True
`.view()` now works, producing shape: torch.Size([2, 5, 6])


## 8. One Last Piece: the Output Projection Layer

After flattening all heads back into a single `(batch, num_tokens, d_out)` tensor, there's one more small addition worth including: a final learned `nn.Linear(d_out, d_out)` layer, applied to the combined result.

**Why add this if concatenation already combines the heads?** Simple concatenation just places every head's numbers *side by side* -- head 1's output occupies the first `head_dim` columns, head 2's occupies the next `head_dim` columns, and so on, with absolutely no mixing between them. A final linear layer lets the model learn to combine information *across* head boundaries -- for example, learning that "head 1's third number together with head 2's first number" is a meaningful combination. This output projection isn't mathematically required for multi-head attention to work, but it's included in essentially every real transformer implementation, so we'll include it here too.

## 9. Building the Efficient `MultiHeadAttention` Class

We now have every piece needed to build a single, efficient module that computes multi-head attention directly, without ever creating separate `CausalAttention` instances:

1. One big `nn.Linear` each for queries, keys, and values (instead of `num_heads` separate small ones).
2. Reshape each with `.view()` to introduce a `num_heads` dimension, splitting the combined output into per-head chunks.
3. `.transpose(1, 2)` to bring `num_heads` next to `batch`, so `@` performs a batched matrix multiplication -- one independent attention computation per head, all at once.
4. Mask with `-inf`, scale, `softmax`, dropout -- identical math to single-head causal attention, just applied across every head simultaneously.
5. `.transpose(1, 2)` back, `.contiguous()`, and `.view()` to flatten the heads back into one `d_out`-sized vector per token.
6. A final `out_proj` linear layer to let the model mix information across heads.

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # size of each individual head's output

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # combines head outputs after concatenation
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)       # (b, num_tokens, d_out) -- ONE matmul for every head at once
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split d_out into (num_heads, head_dim) -- a lossless reshape, not a computation.
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # Bring num_heads next to batch, so `@` batches over (batch, num_heads).
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # One batched matmul computes every head's attention scores at once.
        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        # One batched matmul computes every head's context vectors at once.
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Flatten (num_heads, head_dim) back into a single d_out-sized vector per token.
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # let the model mix information across heads
        return context_vec

Let's run this on our batch, using `num_heads=2` and a *combined* output size of `d_out=4` (so `head_dim = 4 / 2 = 2` -- matching the wrapper example from section 2, where each of the 2 heads individually produced a 2-dimensional output).

In [10]:
torch.manual_seed(123)
d_out = 4  # combined output size across ALL heads
num_heads = 2

mha_efficient = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=num_heads)
context_vecs_efficient = mha_efficient(batch)

print("context_vecs_efficient.shape:", context_vecs_efficient.shape,
      "-> (batch_size, num_tokens, d_out)")
print()
print(context_vecs_efficient)

context_vecs_efficient.shape: torch.Size([2, 5, 4]) -> (batch_size, num_tokens, d_out)

tensor([[[-0.1530,  0.3100, -0.0614, -0.1520],
         [-0.0873,  0.3039, -0.0685, -0.2599],
         [-0.0617,  0.3247, -0.0707, -0.3041],
         [-0.0604,  0.3261, -0.0691, -0.2989],
         [-0.0549,  0.3272, -0.0704, -0.3119]],

        [[-0.1530,  0.3100, -0.0614, -0.1520],
         [-0.0873,  0.3039, -0.0685, -0.2599],
         [-0.0617,  0.3247, -0.0707, -0.3041],
         [-0.0604,  0.3261, -0.0691, -0.2989],
         [-0.0549,  0.3272, -0.0704, -0.3119]]], grad_fn=<ViewBackward0>)


## 10. The Ultimate Proof: Does "Efficient" Really Mean "Identical"?

We've argued, piece by piece, that every trick in `MultiHeadAttention` (the single big matmul, the `.view()` split, the `.transpose()` for batching, `.contiguous()`, flattening back) is just a faster way of computing exactly what running several small, independent attention heads would compute. Let's put that claim to its strongest possible test.

We'll take our trained `mha_efficient` module, but recompute its output *by hand*: pull out its query/key/value projections, manually slice them into per-head chunks with a plain Python loop (no `.view()`, no `.transpose()`, no batched matmul -- just simple indexing and a loop, one head at a time), run the exact same causal-attention math on each slice individually, concatenate the results, and apply `out_proj`. If our understanding is correct, this should match `mha_efficient`'s actual output exactly.

In [11]:
b, num_tok, _ = batch.shape
head_dim = mha_efficient.head_dim
num_heads = mha_efficient.num_heads

# Use the SAME learned layers as mha_efficient -- we're just recomputing its forward
# pass by hand, one head at a time, to double check the efficient version's math.
queries_all = mha_efficient.W_query(batch)  # (b, num_tokens, d_out)
keys_all    = mha_efficient.W_key(batch)
values_all  = mha_efficient.W_value(batch)

context_per_head = []
for h in range(num_heads):
    # Manually slice out this head's chunk of columns -- no .view(), just plain indexing.
    q_h = queries_all[:, :, h*head_dim : (h+1)*head_dim]
    k_h = keys_all[:, :, h*head_dim : (h+1)*head_dim]
    v_h = values_all[:, :, h*head_dim : (h+1)*head_dim]

    # Exactly the single-head causal attention math from the previous notebook.
    attn_scores_h = q_h @ k_h.transpose(1, 2)
    mask_bool = mha_efficient.mask.bool()[:num_tok, :num_tok]
    attn_scores_h = attn_scores_h.masked_fill(mask_bool, -torch.inf)
    attn_weights_h = torch.softmax(attn_scores_h / head_dim**0.5, dim=-1)
    context_h = attn_weights_h @ v_h

    context_per_head.append(context_h)

# Glue the independently-computed heads back together, exactly like the wrapper did.
context_vec_manual = torch.cat(context_per_head, dim=-1)
context_vec_manual = mha_efficient.out_proj(context_vec_manual)

print("Does the hand-computed, per-head result match mha_efficient's actual output?")
print(torch.allclose(context_vec_manual, context_vecs_efficient, atol=1e-6))

Does the hand-computed, per-head result match mha_efficient's actual output?
True


They match exactly. This confirms, with real numbers rather than just an argument, that `MultiHeadAttention`'s reshaping and batched matrix multiplications are not an approximation or a different algorithm -- they compute *precisely* the same thing as running `num_heads` independent single-head attention computations and concatenating their outputs, just packaged into far fewer (and much larger, much faster) matrix multiplications.

## Conclusion

This notebook extended causal self-attention into **multi-head attention**. Here's the "why" behind every new idea, in one line each:

* **Multiple heads** -- one shared set of attention weights can only learn one notion of "relevance"; several independent heads let the model track several different kinds of relationships between tokens at once, then combine all of their views into one richer representation.
* **The wrapper approach** -- the most direct way to get multiple heads: run the input through several independent `CausalAttention` instances and concatenate their outputs. Easy to understand, but wasteful, since it repeats a separate small matrix multiplication for every single head.
* **Split weights (the efficient approach)** -- one big weight matrix, multiplying the input once, produces the exact same numbers as several small weight matrices multiplying it separately -- so we can do all heads' projections in a single, larger, faster matrix multiplication and only split the *result* into per-head chunks afterward.
* **`.view()`** -- a free, lossless reshape (no computation, no data movement) that regroups the combined output's columns into per-head chunks.
* **`.transpose(1, 2)`** -- rearranges dimensions so PyTorch's batched matrix multiplication treats `(batch, num_heads)` as independent matrices to multiply, computing every head's attention scores (and later, context vectors) in one call instead of a Python loop.
* **`.contiguous()`** -- after a transpose, the tensor's shape is reinterpreted but the underlying memory order isn't rearranged; `.view()` needs a fresh, properly-ordered copy of the data to safely reshape, which `.contiguous()` provides.
* **`out_proj`** -- a final learned layer that lets the model mix information *across* heads, since simple concatenation only places their outputs side by side without letting them interact.

We even proved, with a from-scratch, per-head recomputation, that the efficient `MultiHeadAttention` module computes *exactly* the same result as running several independent single-head attention modules by hand -- it's simply a faster way of organizing the identical computation.

This `MultiHeadAttention` module -- causal masking, dropout, and multiple heads, all in one class -- is the actual attention mechanism used inside real GPT-style language models. The smallest version of GPT-2 uses 12 heads with a combined embedding size of 768; the largest uses 25 heads with a combined embedding size of 1,600. The ideas are exactly the ones in this notebook -- just scaled up.